# Lecture 08 — From a fitted model to a service

A model is useful only when the transformation from raw fields to a decision is reproducible. We will package preprocessing and logistic regression together, keep the decision threshold with the fitted model, and call the same model through a validated API. The churn data are synthetic; results are examples, not evidence about real customers.

![Teaching figure for deployment](../../docs/figures/08_deployment.svg)

*Conceptual serving workflow. The same saved preprocessing is applied to each validated incoming record.*

## Serving vocabulary and the artifact lifecycle

An **artifact** is the saved fitted pipeline and its metadata on disk. A server **process** loads it into memory. An **API** is an interface that another program calls. Here an HTTP **request** sends a JSON object (named fields and values) to an **endpoint**, a URL path such as `/predict`; the **response** returns JSON and a status code. Status 200 means the request succeeded; validation failures are a different outcome.

The conceptual chain is raw customer fields → validation → saved preprocessing → fitted probability → stored decision threshold → response. The saved transformations must be the ones learned during training, rather than transformations re-estimated from the incoming customer.

The later `threshold=0.4` artifact is an **illustrative software-contract example**, not the policy selected by the preceding validation exercise. The separate training demonstrations create separate artifacts. A deployed policy must explicitly save the chosen threshold with its corresponding fitted pipeline.

The `try/finally` blocks restore temporary files, environment variables, and caches so the notebook can rerun safely. Read them as execution housekeeping on the first pass. Focus first on saving, loading, predicting, and validating a request. MLflow experiment tracking and Docker packaging are extension topics after that core route.

## 1. Find the project and state the prediction problem

The notebook may be opened from a nested directory. We locate the repository by searching parent folders for `data/churn.csv`, then add `src` for this checkout’s source layout. The response is `churn = 1`; the service returns both the probability and the thresholded decision.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data" / "churn.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate data/churn.csv from the current directory")
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data" / "churn.csv"
print("Project root:", ROOT)
print("Data:", DATA)


In [ ]:
import pandas as pd

data = pd.read_csv(DATA)
print(data.shape)
data.head()

We model \(p(x)=\Pr(Y=1\mid X=x)=\sigma(\beta_0+\beta^T z(x))\), where \(\sigma(t)=(1+e^{-t})^{-1}\) and \(z\) is a learned feature transformation. Numeric standardization centers and rescales each quantitative feature. One-hot encoding gives each contract category its own indicator; this avoids imposing an arbitrary numeric spacing between contract types.

## 2. Train and evaluate once on a held-out sample

The fitting routine uses a stratified random split, fitting preprocessing only on the training part. The held-out rows estimate performance on data not used to choose coefficients. Accuracy measures total correctness, precision is the fraction of predicted churners who churn, recall is the fraction of churners found, F1 balances precision and recall, and ROC AUC measures ranking across cutoffs.

In [ ]:
import tempfile
from course_ml.train import train_model

with tempfile.TemporaryDirectory() as tmp:
    artifact_path = Path(tmp) / "churn.joblib"
    metadata = train_model(DATA, artifact_path, threshold=0.5)
    print("Artifact exists:", artifact_path.is_file())
    display(pd.Series(metadata["metrics"], name="held-out metric"))
    print("Rows:", metadata["train_rows"], "train +", metadata["test_rows"], "test")

## 3. Probability is not the decision

The model estimates probabilities. A separate cutoff \(\tau\) maps them to actions: \(\hat y=1\{p(x)\geq\tau\}\). If missing a churner costs more than contacting a customer who would stay, a smaller cutoff may be appropriate. That cutoff must be selected against a validation set and explicit costs; changing it changes precision and recall without changing the fitted probability model.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from course_ml.train import _load_data, build_pipeline

# First reserve the same 25% final test split used by train_model. Then split
# only the remaining development data into fitting and validation subsets.
x, y = _load_data(DATA)
x_development, x_final_test, y_development, y_final_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)
x_fit, x_validation, y_fit, y_validation = train_test_split(
    x_development, y_development, test_size=0.20, random_state=17, stratify=y_development
)
validation_pipeline = build_pipeline().fit(x_fit, y_fit)
validation_probabilities = validation_pipeline.predict_proba(x_validation)[:, 1]
thresholds = (0.30, 0.50, 0.70)
validation_table = pd.DataFrame([
    {
        "threshold": threshold,
        "precision": precision_score(y_validation, validation_probabilities >= threshold, zero_division=0),
        "recall": recall_score(y_validation, validation_probabilities >= threshold, zero_division=0),
        "f1": f1_score(y_validation, validation_probabilities >= threshold, zero_division=0),
    }
    for threshold in thresholds
]).set_index("threshold")
display(validation_table)
print("Final test rows are held aside:", len(x_final_test))

Each row uses the same validation probabilities and applies a different policy cutoff. Choose a threshold on validation data using the relative cost of missed churn and unnecessary contact. Once the choices are fixed, evaluate once on a separate final test set. A threshold table computed from test labels would leak test information into the model-selection process.

## 4. Inspect the packaged contract

The artifact stores a fitted pipeline and metadata together. The pipeline ensures that serving applies the exact learned scaling and category encoding used at training. Metadata makes the expected schema and decision rule inspectable. The artifact should be treated as executable input because joblib uses pickle serialization.

In [ ]:
import joblib

with tempfile.TemporaryDirectory() as tmp:
    artifact_path = Path(tmp) / "churn.joblib"
    train_model(DATA, artifact_path)
    artifact = joblib.load(artifact_path)
    artifact["metadata"]

## 5. Exercise the HTTP contract without starting a server

FastAPI validates the request before model scoring. `TestClient` invokes the app in process, so the notebook finishes normally and does not leave a server running. A real HTTP client would send the same JSON to a running Uvicorn process.

In [ ]:
import os
import tempfile
from fastapi.testclient import TestClient
from course_ml import app as api

previous_model_path_present = "MODEL_PATH" in os.environ
previous_model_path = os.environ.get("MODEL_PATH")
try:
    with tempfile.TemporaryDirectory() as tmp:
        artifact_path = Path(tmp) / "churn.joblib"
        train_model(DATA, artifact_path, threshold=0.4)
        os.environ["MODEL_PATH"] = str(artifact_path)
        api._load_artifact.cache_clear()
        client = TestClient(api.app)
        health = client.get("/health")
        valid = client.post("/predict", json={
            "tenure_months": 8, "monthly_charge": 92.0,
            "support_calls": 4, "contract": "monthly"
        })
        invalid = client.post("/predict", json={
            "tenure_months": 8, "monthly_charge": 92.0,
            "support_calls": 4, "contract": "quarterly"
        })
        print("health:", health.status_code, health.json())
        print("prediction:", valid.status_code, valid.json())
        print("unknown contract:", invalid.status_code, invalid.json())
finally:
    api._load_artifact.cache_clear()
    if previous_model_path_present:
        os.environ["MODEL_PATH"] = previous_model_path
    else:
        os.environ.pop("MODEL_PATH", None)

## 6. Record and inspect a local experiment

The next cell records an MLflow run in `.verification/tracking-demo/mlflow.db` and writes the model artifact beside it. The ignored directory keeps this demonstration from filling the repository with committed experiment files. It logs parameters, held-out metrics, the joblib artifact, a model signature and example input, the CSV SHA-256 digest, and Python/numerical-library versions. The code restores both the prior `MLFLOW_TRACKING_URI` environment variable and MLflow tracking setting when it finishes. Re-running adds another run to this local experiment.

For the default project tracking database, run `python -m course_ml.train --data data/churn.csv --output artifacts/churn.joblib --track`; MLflow writes to `mlflow.db` by default. The course environment already includes MLflow.

In [ ]:
import mlflow
import os

previous_uri = mlflow.get_tracking_uri()
previous_env_present = "MLFLOW_TRACKING_URI" in os.environ
previous_env_uri = os.environ.get("MLFLOW_TRACKING_URI")
tracking_dir = ROOT / ".verification" / "tracking-demo"
tracking_dir.mkdir(parents=True, exist_ok=True)
database = (tracking_dir / "mlflow.db").as_posix()
tracking_uri = f"sqlite:///{database}"
try:
    os.environ["MLFLOW_TRACKING_URI"] = tracking_uri
    mlflow.set_tracking_uri(tracking_uri)
    artifact_path = tracking_dir / "churn.joblib"
    train_model(DATA, artifact_path, track=True)
    experiment = mlflow.get_experiment_by_name("churn-deployment-lesson")
    runs = mlflow.search_runs([experiment.experiment_id], max_results=5)
    display(runs[["params.threshold", "metrics.roc_auc", "tags.data_sha256"]])
finally:
    if previous_env_present:
        os.environ["MLFLOW_TRACKING_URI"] = previous_env_uri
    else:
        os.environ.pop("MLFLOW_TRACKING_URI", None)
    mlflow.set_tracking_uri(previous_uri)

## 7. Deployment limits

This local teaching API has no authentication, rate limits, TLS setup, audit trail, or privacy controls. Do not expose it publicly or send it real customer records. Even with one shared preprocessing pipeline, live data can differ from training data. A production service should monitor input distributions, missingness, prediction rates, and eventual outcomes; investigate drift before retraining. See the lesson README for the operational and security discussion.

## 8. Statistical estimator and software contract

Statistically, a learning procedure maps a sample \(D\) to an estimated parameter set \(\hat\theta\), and the fitted model maps a feature vector to a probability. The API has a separate software contract: required fields, types and allowed values, response fields, and errors callers can expect. A valid statistical estimator can still be unusable if those rules are unclear or change silently.

Reloading an artifact should preserve predictions for the same inputs in a compatible software environment; our round-trip test checks that property. Reproducing a fit is stronger. It also requires the same data bytes, preprocessing and split rules, random seeds, package versions, and compatible numerical behavior. When hardware or numerical libraries differ, compare fitted predictions with a stated floating-point tolerance.

Covariate shift assumes the feature distribution \(P(X)\) changes while the conditional relationship \(P(Y\mid X)\) stays stable. Under that assumption, monitoring the changed input distribution can guide investigation, but it does not establish that prediction quality fell. If the relationship between features and churn changes, covariate-shift methods do not address that change. Reliable outcome labels are needed to measure predictive quality.